In [2]:
# Configuración inicial
import sys, os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

# Añadir el repo clonado al PYTHONPATH
sys.path.append("./neo4j_graphrag_tutorial")

# Verificar configuración
print("✅ Variables de entorno cargadas desde .env")
print(f"   OPENAI_API_KEY: {'✓ Configurada' if os.getenv('OPENAI_API_KEY') else '✗ No encontrada'}")
print(f"   NEO4J_URI: {os.getenv('NEO4J_URI', '✗ No configurada')}")
print(f"   NEO4J_USERNAME: {os.getenv('NEO4J_USERNAME', '✗ No configurado')}")
print(f"   NEO4J_PASSWORD: {'✓ Configurada' if os.getenv('NEO4J_PASSWORD') and os.getenv('NEO4J_PASSWORD') != 'your_password_here' else '✗ No configurada o placeholder'}")

neo4j_ready = (
    os.getenv('NEO4J_URI') and 
    os.getenv('NEO4J_PASSWORD') and 
    os.getenv('NEO4J_PASSWORD') != 'Mdps2004'
)

if not neo4j_ready:
    print("\n⚠️  ADVERTENCIA: Neo4j no está configurado correctamente")
    print("   Para usar GraphRAG necesitas:")
    print("   1. Neo4j corriendo (Desktop o Aura)")
    print("   2. Configurar credenciales en .env")
    print("   3. Cargar datos con el ETL del tutorial")
else:
    print("\n✅ Neo4j configurado - listo para evaluar GraphRAG")

✅ Variables de entorno cargadas desde .env
   OPENAI_API_KEY: ✓ Configurada
   NEO4J_URI: neo4j://127.0.0.1:7687
   NEO4J_USERNAME: neo4j
   NEO4J_PASSWORD: ✓ Configurada

⚠️  ADVERTENCIA: Neo4j no está configurado correctamente
   Para usar GraphRAG necesitas:
   1. Neo4j corriendo (Desktop o Aura)
   2. Configurar credenciales en .env
   3. Cargar datos con el ETL del tutorial


In [ ]:
# Path setup — permite importar rag_eval desde notebooks/
import sys, os
sys.path.insert(0, os.path.abspath('..'))


# Evaluacion GraphRAG Neo4j -- Sistema Completo

14 evaluadores especializados (6 deterministas + 8 LLM-judge). Comparacion wrapper principal vs naive.

In [ ]:
# Imports
from graphrag_wrapper_standalone import neo4j_graphrag_wrapper_standalone as graphrag_main
from graphrag_wrapper_naive import neo4j_graphrag_naive as graphrag_naive
from graphrag_evaluator_advanced import evaluate_graphrag_advanced, compute_calibration_report
from dataset_northwind_full import DATASET_NORTHWIND
import time

print(f"Dataset: {len(DATASET_NORTHWIND)} preguntas | Evaluadores: 14")

## Dataset (31 preguntas, 7 categorias)

In [ ]:
# Distribución del dataset
slices = [("A. Lookups (1-hop)", 6), ("B. Agregaciones", 8), ("C. Multi-hop 2", 5),
          ("D. Multi-hop 3+", 3), ("E. Sin respuesta", 4), ("F. Filtros complejos", 3), ("G. Jerarquia", 2)]
idx = 0
for cat, n in slices:
    print(f"  {cat}: {n} preguntas")
    idx += n
print(f"Total: {len(DATASET_NORTHWIND)}")

## Evaluacion wrapper principal

In [ ]:
# Evaluacion wrapper principal
import time
ts = int(time.time())
dataset_name = f"northwind-full-{ts}"

results_main = evaluate_graphrag_advanced(
    graphrag_main,
    DATASET_NORTHWIND,
    dataset_name=dataset_name,
    project="graphrag-neo4j-evaluation",
    evaluator_set="full",
    max_concurrency=1,
)
print(f"Experimento: {results_main.experiment_name}")

## Evaluacion wrapper naive (comparacion)

In [ ]:
# Evaluacion wrapper naive (comparacion) -- mismo dataset
results_naive = evaluate_graphrag_advanced(
    graphrag_naive,
    DATASET_NORTHWIND,
    dataset_name=dataset_name,
    project="graphrag-neo4j-evaluation-naive",
    evaluator_set="full",
    max_concurrency=1,
)
print(f"Experimento naive: {results_naive.experiment_name}")

## Comparacion Principal vs Naive

In [ ]:
# Comparar ambos sistemas
from langsmith import Client
client = Client()

def get_avg_scores(exp_name):
    runs = list(client.list_runs(project_name=exp_name, run_type="chain"))
    raw = {}
    for run in runs:
        for fb in client.list_feedback(run_ids=[run.id]):
            if fb.score is not None:
                raw.setdefault(fb.key, []).append(fb.score)
    return {k: sum(v)/len(v) for k, v in raw.items() if v}

ms = get_avg_scores(results_main.experiment_name)
ns = get_avg_scores(results_naive.experiment_name)

metrics = [
    ("cypher_generated",            "Cypher generado"),
    ("cypher_result_nonempty",      "Resultados no vacios"),
    ("empty_context_hallucination", "Sin alucinacion silenciosa"),
    ("schema_adherence",            "Schema adherence"),
    ("relationship_direction_score","Direccion relaciones"),
    ("structural_score",            "Score estructural"),
    ("correctness",                 "Correctness"),
    ("graphrag_groundedness",       "Groundedness"),
    ("answer_completeness",         "Completeness"),
    ("failure_mode",                "Sin fallo detectado"),
]

print(f"{'METRICA':<35} {'PRINCIPAL':>10} {'NAIVE':>10} {'DELTA':>10}")
print("-" * 68)
for key, label in metrics:
    m, n = ms.get(key), ns.get(key)
    if m is None and n is None: continue
    ms2 = f"{m:.3f}" if m else "N/A"
    ns2 = f"{n:.3f}" if n else "N/A"
    ds  = f"{m-n:+.3f}" if (m and n) else "N/A"
    win = " +" if (m and n and m-n > 0.01) else (" -" if (m and n and m-n < -0.01) else " =")
    print(f"{label:<35} {ms2:>10} {ns2:>10} {ds:>10}{win}")

def conf(s):
    c = s.get("correctness", 0.5)
    g = s.get("graphrag_groundedness", 0.5)
    st = s.get("structural_score", 0.5)
    co = s.get("answer_completeness", 0.5)
    r = s.get("relevance", 0.5)
    q = 0.4*c + 0.3*g + 0.2*co + 0.1*r
    return round(0.4*st + 0.6*q, 3)

cm, cn = conf(ms), conf(ns)
print("-" * 68)
print(f"{'CONFIDENCE SCORE V2':<35} {cm:>10.3f} {cn:>10.3f} {cm-cn:>+10.3f}")
print(f"Principal supera al naive en {(cm-cn)*100:.1f} puntos porcentuales")

## Calibracion ECE

In [ ]:
# Calibracion ECE
from langsmith import Client
client = Client()

conf_scores, acc_labels = [], []
runs = list(client.list_runs(project_name=results_main.experiment_name, run_type="chain"))
for run in runs:
    fb_map = {}
    for fb in client.list_feedback(run_ids=[run.id]):
        if fb.score is not None:
            fb_map[fb.key] = fb.score
    if "correctness" in fb_map and "structural_score" in fb_map:
        c  = fb_map.get("correctness", 0.5)
        g  = fb_map.get("graphrag_groundedness", 0.5)
        s  = fb_map.get("structural_score", 0.5)
        co = fb_map.get("answer_completeness", 0.5)
        r  = fb_map.get("relevance", 0.5)
        h  = fb_map.get("empty_context_hallucination", 1.0)
        q  = 0.4*c + 0.3*g + 0.2*co + 0.1*r
        conf = max(0.0, min(1.0, 0.4*s + 0.6*q - (0.40 if h == 0 else 0.0)))
        conf_scores.append(round(conf, 3))
        acc_labels.append(int(fb_map.get("correctness", 0)))

if conf_scores:
    report = compute_calibration_report(conf_scores, acc_labels)
    print(f"CALIBRATION REPORT (n={report['n_samples']})")
    print(f"  ECE: {report['ece']:.4f}  --  {report['interpretation']}")
    print(f"  Accuracy:   {report['overall_accuracy']:.3f}")
    print(f"  Confidence: {report['overall_confidence']:.3f}")
    print(f"  Gap:        {report['confidence_gap']:+.3f}")
    for b in report["bins"]:
        print(f"  {b['range']}: acc={b['accuracy']:.2f} conf={b['avg_confidence']:.2f} gap={b['gap']:.2f} (n={b['n']})")

---
## Evaluación Universal (cualquier RAG)
Usa `universal_rag_evaluator.py` — métricas académicas universales:
- **faithfulness_nli**: DeBERTa NLI sin LLM (TRUE/RAGAS paper, NAACL/EACL 2022/2024)
- **atomic_fact_precision**: FActScore-style decomposición en hechos atómicos + NLI (EMNLP 2023)
- **context_precision_at_k**: Ranking ponderado de chunks relevantes (RAGAS, EACL 2024)
- **context_recall**: Cobertura del GT por el contexto (RAGAS, EACL 2024)
- **negative_rejection**: Anti-alucinación en preguntas sin respuesta (RGB, AAAI 2024)
- **answer_relevance_universal**: LLM-judge G-Eval style (EMNLP 2023)
- **correctness_universal**: Correctness vs ground truth (LLM-judge)
- **confidence_score_universal**: Score compuesto calibrado (inspirado en ARES, NAACL 2024)

Compatible con **cualquier RAG** que implemente `fn(inputs: dict) -> dict` con al menos `{answer, context}`.

In [ ]:
# Imports universal evaluator
from universal_rag_evaluator import (
    faithfulness_nli,
    hallucination_rate,
    atomic_fact_precision,
    context_precision_at_k,
    context_recall,
    context_relevance,
    answer_relevance_universal,
    correctness_universal,
    negative_rejection,
    confidence_score_universal,
    evaluate_rag_universal,
    print_universal_summary,
    DEFAULT_EVALUATORS,
    FULL_EVALUATORS,
    NLI_ONLY_EVALUATORS,
    compute_ece,
    compute_calibration_report,
    find_optimal_temperature,
)
from graphrag_wrapper_standalone import neo4j_graphrag_wrapper_standalone as graphrag_main
from dataset_northwind_full import DATASET_NORTHWIND
import time

print(f'Dataset: {len(DATASET_NORTHWIND)} preguntas')
print('Evaluadores universales disponibles:')
for ev in FULL_EVALUATORS:
    print(f'  - {ev.__name__}')

### Preset: `default` — métricas esenciales sin sobrecarga
Usa `faithfulness_nli` (NLI, sin LLM), `correctness_universal` + `answer_relevance_universal` (LLM-judge),
`negative_rejection` (determinista) y `confidence_score_universal` (compuesto).

In [ ]:
# Evaluación universal — preset 'default'
# Compatible con el GraphRAG wrapper existente
ts = int(time.time())
universal_dataset_name = f'northwind-universal-{ts}'

results_universal = evaluate_rag_universal(
    rag_fn=graphrag_main,
    dataset=DATASET_NORTHWIND,
    dataset_name=universal_dataset_name,
    project='graphrag-neo4j-evaluation',
    preset='default',
    max_concurrency=1,
)
print(f'Experimento: {results_universal.experiment_name}')

In [ ]:
# Resumen de métricas universales
metrics_universal = print_universal_summary(
    results_universal,
    title='Universal RAG Evaluation — GraphRAG Principal'
)

### Preset: `nli_only` — solo métricas NLI (costo $0, sin LLM)
Útil para: monitorización en producción, screening rápido, privacidad de datos.

In [ ]:
# Evaluación solo con NLI (sin ninguna llamada LLM)
ts2 = int(time.time())
nli_dataset_name = f'northwind-nli-only-{ts2}'

results_nli = evaluate_rag_universal(
    rag_fn=graphrag_main,
    dataset=DATASET_NORTHWIND,
    dataset_name=nli_dataset_name,
    project='graphrag-neo4j-evaluation',
    preset='nli_only',
    max_concurrency=1,
)
print(f'Experimento NLI-only: {results_nli.experiment_name}')

### Calibración ECE de los evaluadores universales
Calcula el ECE del `correctness_universal` usando `faithfulness_nli` como predictor de confianza.
Si ECE > 0.10 → aplicar temperature scaling para calibrar.

**Referencias**: 'When Can We Trust LLM Graders?' (arXiv:2603.29559, 2025); Guo et al. ICML 2017.

In [ ]:
# Calibración ECE del evaluador universal
from langsmith import Client
client = Client()

conf_scores_u, acc_labels_u = [], []
runs_u = list(client.list_runs(
    project_name=results_universal.experiment_name, run_type='chain'
))

for run in runs_u:
    fb_map = {}
    for fb in client.list_feedback(run_ids=[run.id]):
        if fb.score is not None:
            fb_map[fb.key] = fb.score

    if 'correctness_universal' in fb_map and 'faithfulness_nli' in fb_map:
        # faithfulness_nli como proxy de confianza
        conf = fb_map['faithfulness_nli']
        acc = int(fb_map['correctness_universal'])
        conf_scores_u.append(conf)
        acc_labels_u.append(acc)

if conf_scores_u:
    print(f'Pares disponibles para calibración: {len(conf_scores_u)}')
    report = compute_calibration_report(conf_scores_u, acc_labels_u, metric_name='faithfulness_nli')

    # Temperatura óptima
    opt = find_optimal_temperature(conf_scores_u, acc_labels_u)
    print(f'\nTemperatura óptima: T={opt["optimal_temperature"]}')
    print(f'ECE antes: {opt["ece_before"]} → ECE después: {opt["ece_after"]} (mejora: {opt["improvement"]})')
    print(f'Scores calibrados (primeros 5): {opt["calibrated_scores"][:5]}')
else:
    print('No hay suficientes pares para calibración. Ejecuta primero la evaluación universal.')

## 🔧 Troubleshooting

Si obtienes errores, verifica:

### 1. Neo4j está corriendo
```bash
# Para Neo4j Desktop: verifica que esté iniciado
# Para Neo4j Aura: verifica la conexión
```

### 2. Credenciales correctas en `.env`
```bash
NEO4J_URI=bolt://localhost:7687  # o tu URI de Aura
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=tu_password_real  # NO "your_password_here"
```

### 3. Datos cargados en Neo4j
```bash
cd neo4j_graphrag_tutorial
python ETL_pipelines/main_etl.py
```

### 4. Dependencias instaladas
```bash
pip install langchain langchain-community langchain-openai neo4j python-dotenv
```

Si todo falla, ejecuta el script de debugging:
```bash
python debug_graphrag.py
```